In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_crafting_table():
    url = "https://darkanddarker.wiki.spellsandguns.com/Crafting"
    
    with requests.Session() as ses:
        ses.headers.update({"User-Agent": "Mozilla/5.0"})
        response = ses.get(url)
        response.raise_for_status()
    
    soup = BeautifulSoup(response.text, "html.parser")
    recipes = []

    for table in soup.find_all("table"):
        for row in table.find_all("tr")[1:]:  # skip header row
            cols = row.find_all("td")
            if len(cols) < 3:
                continue

            # Name cell — grab the bold link text
            name_cell = cols[0]
            bold = name_cell.find("b")
            name = bold.get_text(strip=True) if bold else name_cell.get_text(strip=True)

            # Ingredients cell — multiple ingredients separated by <br> or just text
            ingredients_cell = cols[1]
            ingredients = ingredients_cell.get_text(separator="|", strip=True)

            # Merchant
            merchant = cols[2].get_text(strip=True)

            # Affinity (may be empty)
            affinity = cols[3].get_text(strip=True) if len(cols) > 3 else ""

            recipes.append({
                "name": name,
                "ingredients": ingredients,
                "merchant": merchant,
                "affinity": affinity,
            })

    return recipes

recipes = scrape_crafting_table()
for r in recipes:
    print(r)